# Ch 5: Cross-language passage content comparison (EN vs FR)

Systematic comparison of how fiction's content composition shifts across 1600–1800 in English vs French. Data: qwen3.5-35b-a3b passage-content annotations (V3), joined to lltk.texts for year/lang. Parallel and divergent trends along:
- **scene_content** (16 types): courtship, battle, moral_reflection, domestic_routine, etc.
- **setting** (7 types): domestic_interior, grand_estate, wilderness, etc.
- **character_classes** (6): noble_aristocratic, gentry_or_middling, etc.
- **fantastical_elements** (6): supernatural, ghost_or_haunting, etc.
- **abstractness** (continuous score) for context

## Corpus caveat

The French content annotations come almost entirely (99.7%) from `_gallica_literary_fictions` (503 texts). The English side draws from chadwyck, earlyprint, litlab, etc. This means cross-language differences reflect **canon composition** — what each tradition anthologizes as 'literary fiction' — rather than a random sample of all fiction in each language. That's still a meaningful comparative claim, but it's a claim about canons, not about all fiction.

In [ ]:
import clickhouse_connect
import pandas as pd
import matplotlib.pyplot as plt

from abstraction.aggregate import passage_abstractness_by_lang
from abstraction.analysis import fetch_cross_language_fields
from abstraction.plotting import plot_cross_language_lines

plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 200

CH = dict(host='localhost', port=8123, username='lltk', password='lltk')
c = clickhouse_connect.get_client(**CH)

## Helper sanity check

`fetch_cross_language_fields` returns long-form (lang, period, field, pct, n); `plot_cross_language_lines` renders a horizontal strip of EN-vs-FR panels. Both live in `abstraction` so we can reuse them from other notebooks.

In [ ]:
_test = fetch_cross_language_fields([
    ('fantastical_elements', 'supernatural'),
    ('fantastical_elements', 'prophecy_or_omen'),
], client=c)
print(_test.pivot_table(index=['field','period'], columns='lang', values='pct').round(1))

## 1. Parallel trends: secular domesticity (both languages)

The novel's turn inward — toward domestic interiors, domestic routines, and middling characters — is *shared* across EN and FR, with EN climbing more steeply.

In [ ]:
parallel_fields = [
    ('scene_content', 'domestic_routine'),
    ('setting', 'domestic_interior'),
    ('character_classes', 'gentry_or_middling'),
    ('scene_content', 'battle_or_violence'),
]

df_parallel = fetch_cross_language_fields(parallel_fields, client=c)
print(df_parallel.pivot_table(index=['field', 'period'], columns='lang', values='pct').round(1))

In [ ]:
plot_cross_language_lines(
    df_parallel, parallel_fields,
    title='Parallel trends: EN and FR both secularize toward domesticity',
    figname='ch5_parallel_domesticity.png',
    figsize=(16, 3.8),
)
plt.show()

## 2. Divergent trend 1: the supernatural (EN secularizes, FR re-enchants)

The single most striking cross-language divergence. English fantastical/supernatural content drops ~3x across 1600-1800 (12.5% → 3.9%). French supernatural has a U-shape, dipping mid-17C then climbing above its 17C level by 1750-1800 (11.1% → 13.8%). The French curve is driven by the conte merveilleux / oriental tale / fantastic voyage tradition: Galland's *Mille et Une Nuits* (1704), Fénelon's *Télémaque* (1699→many editions), Mouhy's *Lamékis*, Crébillon's *Ah quel conte!*, the *Cabinet des fées* compilations (1785–89).

In [ ]:
supernatural_fields = [
    ('fantastical_elements', 'supernatural'),
    ('fantastical_elements', 'prophecy_or_omen'),
    ('fantastical_elements', 'ghost_or_haunting'),
    ('fantastical_elements', 'dream_or_vision'),
    ('fantastical_elements', 'allegorical_personification'),
]
df_sup = fetch_cross_language_fields(supernatural_fields, client=c)
print(df_sup.pivot_table(index=['field', 'period'], columns='lang', values='pct').round(1))

In [ ]:
plot_cross_language_lines(
    df_sup, supernatural_fields,
    title='Fantastical content: EN secularizes, FR preserves/re-grows the merveilleux',
    figname='ch5_supernatural_divergence.png',
    figsize=(18, 3.8),
)
plt.show()

### The French supernatural texts (which canon entries drive the U-shape?)

In [ ]:
q = """
SELECT a._id, any(t.year) AS year, any(t.author) AS author, any(t.title) AS title,
       count() AS n_psg,
       countIf(has(JSONExtract(a.value, 'Array(String)'), 'supernatural')) AS n_sup,
       round(100.0 * countIf(has(JSONExtract(a.value, 'Array(String)'), 'supernatural')) / count(), 1) AS sup_pct
FROM llmtasks.passage_annotations_latest a
JOIN (SELECT _id, lang, year, author, title FROM lltk.texts FINAL) t ON a._id = t._id
WHERE a.source_agent = 'qwen3.5-35b-a3b' AND a.task = 'passage-content'
  AND a.field = 'fantastical_elements' AND t.lang = 'fr'
  AND t.year >= 1700 AND t.year < 1800
GROUP BY a._id
HAVING n_sup >= 3
ORDER BY n_sup DESC
LIMIT 20
"""
top_sup = c.query_df(q)
top_sup['title_short'] = top_sup['title'].str.slice(0, 60)
print(top_sup[['year','author','title_short','n_sup','n_psg','sup_pct']].to_string(index=False))

## 3. Divergent trend 2: the moral reflection gap

English moral_reflection stays stable/rises slightly. French moral_reflection *declines* from 59.7% in early 17C to ~54% by late 18C. The interpretation isn't straightforward: French fiction doesn't become *less* moral, but moral commentary gets distributed differently — possibly into character interiority rather than explicit moralizing, aligning with sentimental and psychological fiction.

In [ ]:
reflection_fields = [
    ('scene_content', 'moral_reflection'),
    ('scene_content', 'religious_content'),
    ('character_classes', 'clergy'),
]
df_refl = fetch_cross_language_fields(reflection_fields, client=c)

plot_cross_language_lines(
    df_refl, reflection_fields,
    title='Moral / religious discourse: diverging explicit framing',
    figname='ch5_moral_divergence.png',
    figsize=(13, 3.8),
)
plt.show()

## 4. Divergent trend 3: class composition (the bourgeois novel is English)

English gentry/middling characters rise from ~50% to ~71% (+21pts) across 1600–1800 — a steep middle-class turn. French stays roughly flat (~60%→68%, +3pts only from 1650 baseline). French noble_aristocratic peaks at 85% in the mid-17C (Grand Siècle) and declines, but *relatively* the aristocracy retains much more presence in French fiction than in English.

In [ ]:
class_fields = [
    ('character_classes', 'noble_aristocratic'),
    ('character_classes', 'gentry_or_middling'),
    ('character_classes', 'laboring_or_servant'),
    ('character_classes', 'criminal_or_underclass'),
]
df_class = fetch_cross_language_fields(class_fields, client=c)

plot_cross_language_lines(
    df_class, class_fields,
    title='Class composition: the bourgeois turn is English',
    figname='ch5_class_divergence.png',
    figsize=(16, 3.8),
)
plt.show()

## 5. Courtship: persistent French lead

Courtship is more prevalent in French fiction throughout the period, though the gap narrows over time. This reflects the centrality of *amour-galant*, the *roman sentimental*, and libertine fiction to French literary history.

In [ ]:
courtship_fields = [
    ('scene_content', 'courtship'),
    ('scene_content', 'sexual_encounter'),
    ('scene_content', 'wedding_or_marriage_ceremony'),
    ('scene_content', 'ball_or_social_gathering'),
]
df_crt = fetch_cross_language_fields(courtship_fields, client=c)

plot_cross_language_lines(
    df_crt, courtship_fields,
    title='Courtship and sociability: French lead narrows',
    figname='ch5_courtship.png',
    figsize=(16, 3.8),
)
plt.show()

## 6. Abstractness trends alongside content (the key Ch 5 chart)

Cross-referencing: does the supernatural re-enchantment in French track with higher abstractness? Does the domestic turn in English correlate with rising concreteness?

In [ ]:
abs_df = passage_abstractness_by_lang(ch_client=c)
# Make period int for plotting (start of bin)
abs_df['period_start'] = abs_df['period'].str.split('-').str[0].astype(int)
print(abs_df.pivot(index='period', columns='lang', values='mean_abs').round(3))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: abstractness (abs_df has period as string '1600-1649' + period_start int)
for lang, color in [('en', '#1f77b4'), ('fr', '#d62728')]:
    d = abs_df[abs_df['lang'] == lang].sort_values('period')
    ax1.plot(d['period_start'] + 25, d['mean_abs'], 'o-', color=color, label=f'{lang.upper()} (N={d["n"].sum():,})', linewidth=2.5, markersize=8)
    ax1.fill_between(d['period_start'] + 25,
                     d['mean_abs'] - 1.96 * d['se_abs'],
                     d['mean_abs'] + 1.96 * d['se_abs'],
                     color=color, alpha=0.15)
ax1.set_title('Mean passage abstractness', fontsize=12)
ax1.set_xlabel('year (bin center)')
ax1.set_ylabel('abstractness (+ = more abstract)')
ax1.grid(alpha=0.3)
ax1.legend(fontsize=9)

# Right: supernatural (df_sup has period as int)
sup_df = df_sup[df_sup['field'] == 'fantastical_elements__supernatural']
for lang, color in [('en', '#1f77b4'), ('fr', '#d62728')]:
    d = sup_df[sup_df['lang'] == lang].sort_values('period')
    ax2.plot(d['period'] + 25, d['pct'], 'o-', color=color, label=f'{lang.upper()}', linewidth=2.5, markersize=8)
ax2.set_title('% supernatural passages', fontsize=12)
ax2.set_xlabel('year (bin center)')
ax2.set_ylabel('% of passages')
ax2.grid(alpha=0.3)
ax2.legend(fontsize=9)

fig.suptitle('Abstractness and supernatural: the Ch 5 cross-language pair', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../figures/ch5_abstractness_supernatural.png', bbox_inches='tight')
plt.show()

## 7. Summary table — effect sizes (1600s vs 1750s)

One compact table for the chapter.

In [ ]:
all_df = pd.concat([df_parallel, df_sup, df_refl, df_class, df_crt]).drop_duplicates(['lang','field','period'])

wide = all_df.pivot_table(index=['lang','field'], columns='period', values='pct').reset_index()
wide['delta_1600_1750'] = wide[1750] - wide[1600]

# Reshape to: one row per field, columns (EN_1600, EN_1650, ..., EN_delta, FR_1600, ..., FR_delta)
summary = wide.melt(id_vars=['lang','field'], var_name='metric', value_name='val')
summary['col'] = summary['lang'].str.upper() + '_' + summary['metric'].astype(str)
summary = summary.pivot(index='field', columns='col', values='val')
ordered = [f'EN_{p}' for p in [1600,1650,1700,1750]] + ['EN_delta_1600_1750'] + \
          [f'FR_{p}' for p in [1600,1650,1700,1750]] + ['FR_delta_1600_1750']
summary = summary[[c for c in ordered if c in summary.columns]]
summary['|Δ_EN - Δ_FR|'] = (summary['EN_delta_1600_1750'] - summary['FR_delta_1600_1750']).abs()
print(summary.sort_values('|Δ_EN - Δ_FR|', ascending=False).round(1).to_string())

In [ ]:
summary.to_csv('../figures/ch5_cross_language_summary.csv')
print('Saved to figures/ch5_cross_language_summary.csv')

In [ ]:
c.close()